# SSA MEC Scatter Plot Analysis

Load **monthly gridded** AeroCom pickles, define regions with `create_region_mask()`,
aggregate with `regional_aggregate()`, and compute MEC / MAC / SSA in the notebook.

Regions: **global**, **africa**, **amazon**, **outflow_af** (same boxes and fire seasons as before).

Two plot sets:
1. **Regional mean** scatter plots (one point per model)
2. **Monthly** scatter / time-series plots (one point per model per month)


## 1. Setup System Path and Import Modules

In [ ]:
import sys
import os

# Add parent directory to system path to allow imports
notebook_dir = os.path.dirname(os.path.abspath('SSA_MEC_scatter_plot.ipynb'))
project_root = os.path.dirname(notebook_dir)
py_dir = os.path.join(project_root, 'py')

# Add paths to sys.path if not already present
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if py_dir not in sys.path:
    sys.path.insert(0, py_dir)

print(f"Project root: {project_root}")
print(f"Python directory: {py_dir}")
print(f"System path updated: {sys.path[:3]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import pandas as pd
import xarray as xr
import cftime
from pathlib import Path

# Import project modules
try:
    from py import functions
    from py import cameo_toolbox as ct
    from py import aerocom_data
    print('Successfully imported functions, cameo_toolbox and aerocom_data')
except ImportError:
    import functions
    import cameo_toolbox as ct
    import aerocom_data
    print('Successfully imported functions, cameo_toolbox and aerocom_data (from py dir)')


## 2. Define Data Paths

Monthly gridded pickles are produced by `py/get_data.py` in `Data/var_files/original/monthly/`.


In [ ]:
monthly_pickle = Path(project_root) / 'Data' / 'var_files' / 'original' / 'monthly' / 'monthly_aerocom_data.pickle'
processed_monthly_dir = Path(project_root) / 'Data' / 'AP3_processed_monthly'
land_mask_path = Path(project_root) / 'Data' / 'AEROCOM_III' / 'CAM5.3-Oslo_AP3-CTRL2016-PD' / 'aerocom3_CAM5.3-Oslo_AP3-CTRL2016-PD_landf_Surface_2010_monthly.nc'

print(f'Monthly master pickle: {monthly_pickle}')
print(f'  exists: {monthly_pickle.exists()}')
print(f'Processed monthly NetCDF directory: {processed_monthly_dir}')
print(f'  exists: {processed_monthly_dir.exists()}')
print(f'Land mask file: {land_mask_path}')
print(f'  exists: {land_mask_path.exists()}')

if processed_monthly_dir.exists():
    n_folders = len([d for d in processed_monthly_dir.iterdir() if d.is_dir()])
    print(f'\nProcessed monthly sub-folders: {n_folders}')


## 3. Load the new monthly AeroCom master pickle

The master pickle contains a dictionary `data[model][variable] = xr.Dataset` with all monthly variables.
Base aerosol variables are loaded first; MEC, MAC, SSA and AE are derived afterwards using `aerocom_data.calculate_derived_var`.


### Alternative: load from NetCDF instead of pickle

The monthly data is now also saved as structured NetCDF files under `Data/AP3_processed_monthly/<variable>/<model>_<variable>_processed.nc`. You can reconstruct the same `data[model][variable] = xr.Dataset` dictionary directly from these files using `aerocom_data.load_monthly_data_from_netcdf`, avoiding the legacy master pickle entirely.


In [ ]:
# Example: reconstruct the monthly dictionary from NetCDF files
# (uncomment the assignment below to use it instead of the pickle load above)
# monthly_data = aerocom_data.load_monthly_data_from_netcdf(
#     output_base_dir=str(processed_monthly_dir)
# )
# print(f'NetCDF-based loader: {len(monthly_data)} models available')


In [ ]:
def load_monthly_pickle(path):
    """Load the monthly AeroCom master pickle."""
    try:
        with open(path, 'rb') as f:
            return pickle.load(f)
    except Exception as e:
        print(f'Error loading {path}: {e}')
        return None


monthly_data = load_monthly_pickle(monthly_pickle)
if monthly_data is None:
    raise FileNotFoundError(f'Could not load {monthly_pickle}. Run the monthly extraction first.')

print(f"Loaded {len(monthly_data)} models from the master pickle.")
for m in list(monthly_data.keys())[:5]:
    nvars = sum(1 for v in monthly_data[m].values() if v is not None)
    print(f"  {m}: {nvars} variables available")


In [ ]:
monthly_data

## 4. Examine loaded data structure


In [ ]:
sample_model = next(iter(monthly_data.keys()))
sample_var = next(v for v, ds in monthly_data[sample_model].items() if ds is not None)
sample_da = monthly_data[sample_model][sample_var][sample_var]
print(f'Sample model: {sample_model}')
print(f'Sample variable: {sample_var}')
print(f'Dims: {sample_da.dims}, shape: {sample_da.shape}')
print(f'Time range: {str(sample_da.time.values[0])[:10]} to {str(sample_da.time.values[-1])[:10]}')
print(f'Lat: {float(sample_da.lat.min()):.1f} to {float(sample_da.lat.max()):.1f}')
print(f'Lon: {float(sample_da.lon.min()):.1f} to {float(sample_da.lon.max()):.1f}')


## 5. Compute derived variables and regional means

Workflow:
1. Load all monthly base variables for every model.
2. Convert cftime axes to `datetime64`, normalise monthly timestamps to first-of-month, and shift any model with a -180..180 longitude grid to 0..360 so that all models share the same regional boxes.
3. Calculate derived diagnostics (MEC, MAC, SSA, AE) with `aerocom_data.calculate_derived_var`.
4. Build regional masks with `ct.create_region_mask`. The `SURFACE_TYPE` parameter lets you choose **all**, **land** or **ocean** pixels. The land/ocean mask is taken from the CAM5.3-Oslo land-fraction file and interpolated to each model grid.
5. Apply `ct.regional_aggregate` to the derived fields (not to the base variables) to obtain regional means.

Note: regions that contain no land/ocean grid cells for the requested `SURFACE_TYPE` will produce NaN values (e.g. African outflow is almost entirely ocean, so `land` returns NaN; Africa itself is almost entirely land, so `ocean` returns NaN).


In [ ]:
# --- Configuration ---
# List of model names to exclude from all loops, regressions, and plots.
# Names must match the keys in the monthly pickle exactly.
EXCLUDE_MODELS = []  # e.g., ['GEOS-i33p2-met2010_AP3-CTRL', 'MIROC-SPRINTARS_AP3-CTRL']

SURFACE_TYPE = 'all'  # 'all', 'land', or 'ocean'
LAND_MASK_PATH = land_mask_path  # set to None to disable land/ocean filtering

# Apply model exclusion list before any processing
missing_excluded = [m for m in EXCLUDE_MODELS if m not in monthly_data]
if missing_excluded:
    print(f'Warning: excluded models not found in data: {missing_excluded}')
actually_excluded = [m for m in EXCLUDE_MODELS if m in monthly_data]
monthly_data = {m: v for m, v in monthly_data.items() if m not in EXCLUDE_MODELS}
print(f'Models after exclusion: {len(monthly_data)} (excluded {len(actually_excluded)}: {actually_excluded})')

REGIONS = {
    'global': {
        'lon_range': (0, 360), 'lat_range': (-90, 90),
        'time_slice': ('2010-01-01', '2010-12-31'), 'edge_weighted': False,
    },
    'africa': {
        'lon_range': (15, 37), 'lat_range': (-15, 0),
        'time_slice': ('2010-06-01', '2010-09-30'), 'edge_weighted': False,
    },
    'amazon': {
        'lon_range': (287, 317), 'lat_range': (-17, -3),
        'time_slice': ('2010-07-01', '2010-10-31'), 'edge_weighted': False,
    },
    'outflow_af': {
        'lon_range': (350, 8), 'lat_range': (-15, 3),
        'time_slice': ('2010-06-01', '2010-09-30'), 'edge_weighted': True,
    },
}

BASE_VARS = [
    'od550aer', 'abs550aer', 'od440aer', 'od870aer', 'od865aer',
    'loadbc', 'loaddust', 'loadoa', 'loadso4', 'loadss',
    'emibc', 'emidust', 'emioa', 'emiso2', 'emiss',
]
DERIVED_VARS = ['MEC', 'MAC', 'SSA', 'AE']


def normalize_dataset(ds):
    """Convert cftime time axes to datetime64 and leave everything else untouched."""
    if ds is None:
        return None
    if not isinstance(ds, (xr.Dataset, xr.DataArray)):
        return ds
    if 'time' not in ds.coords or len(ds.time) == 0:
        return ds
    if isinstance(ds, xr.Dataset):
        return functions.convert_cftime_to_datetime(ds)
    if isinstance(ds.time.values[0], cftime.datetime):
        return ds.assign_coords(time=np.array(ds.time.values, dtype='datetime64[ns]'))
    return ds


def extract_dataarray(ds, var_name):
    """Extract a DataArray from a Dataset or pass a DataArray through."""
    if ds is None:
        return None
    if isinstance(ds, xr.DataArray):
        return ds
    if isinstance(ds, xr.Dataset):
        if var_name in ds.data_vars:
            return ds[var_name]
        data_vars = list(ds.data_vars)
        if len(data_vars) == 1:
            return ds[data_vars[0]]
    return None


def ensure_lon_360(da):
    """Shift longitudes from -180..180 to 0..360 so all models share the same convention."""
    if 'lon' in da.coords and float(da.lon.min()) < 0:
        return functions.shift360(da)
    return da


# 1. Load all model data into a flat dict of DataArrays
model_data = {}
for model, var_dict in monthly_data.items():
    base = {}
    for v in BASE_VARS:
        ds = normalize_dataset(var_dict.get(v))
        if ds is not None:
            base[v] = ds

    # 2. Compute derived variables using the vectorized helper
    for dv in DERIVED_VARS:
        ds = aerocom_data.calculate_derived_var(base, model, dv)
        if ds is not None and dv in ds:
            base[dv] = normalize_dataset(ds)

    # 3. Flatten to DataArrays, normalise monthly timestamps and shift longitudes to 0-360
    flat = {}
    for v, ds in base.items():
        da = extract_dataarray(ds, v)
        if da is not None:
            flat[v] = ensure_lon_360(functions.normalize_monthly_time(da))

    if flat:
        model_data[model] = flat

print(f"Models with usable data: {len(model_data)}")


# 4. Build regional masks for every model on its own grid

def get_template(flat):
    """Pick a 2-D spatial template from a model's available variables."""
    for v in ['od550aer', 'MEC', 'SSA', 'MAC']:
        da = flat.get(v)
        if da is not None:
            return da.isel(time=0, drop=True)
    return None


region_masks = {}
for model, flat in model_data.items():
    template = get_template(flat)
    if template is None:
        print(f'No spatial template available for {model}; skipping masks.')
        continue

    region_masks[model] = {}
    for name, cfg in REGIONS.items():
        region_masks[model][name] = ct.create_region_mask(
            template,
            region=name,
            surface_type=SURFACE_TYPE,
            land_mask_path=LAND_MASK_PATH if SURFACE_TYPE in ('land', 'ocean') else None,
            mask_registry=region_masks[model],
        )

print(f"Built masks for {len(region_masks)} models (surface={SURFACE_TYPE}).")


# 5. Aggregate derived variables over each region

def aggregate_region(var_name, region_name, return_time_series=False):
    """Area-weighted regional mean of a derived variable for every model."""
    cfg = REGIONS[region_name]
    result = {}
    for model, flat in model_data.items():
        da = flat.get(var_name)
        if da is None or model not in region_masks:
            continue
        try:
            result[model] = ct.regional_aggregate(
                da,
                region_masks[model][region_name],
                spatial='mean',
                edge_weighted=cfg['edge_weighted'],
                time_slice=cfg['time_slice'],
                temporal='mean',
                return_time_series=return_time_series,
            )
        except Exception as e:
            print(f"Skipping {model} {region_name} {var_name}: {e}")
    return result


MEC = {r: aggregate_region('MEC', r) for r in REGIONS}
SSA = {r: aggregate_region('SSA', r) for r in REGIONS}
MAC = {r: aggregate_region('MAC', r) for r in REGIONS}


# 6. Assemble regional mean DataFrames for plotting

def extract_region_data(region_name, **var_dicts):
    """Build a DataFrame of common models for a given region."""
    available = {name: d for name, d in var_dicts.items() if region_name in d}
    if not available:
        return None
    common_models = sorted(set.intersection(
        *[set(d[region_name].keys()) for d in available.values()]
    ))
    if not common_models:
        return None
    data = {'model': common_models}
    for name, d in available.items():
        data[name] = [d[region_name][m] for m in common_models]
    return pd.DataFrame(data)


region_data = {}
for region in REGIONS:
    df = extract_region_data(region, SSA=SSA, MEC=MEC, MAC=MAC)
    if df is not None:
        region_data[region] = df
        print(f"{region}: {len(df)} models")


# 7. Build monthly time series (one point per model per month)

def build_monthly_region_data(regions):
    """Monthly spatial means of derived variables (no temporal mean)."""
    monthly = {}
    for region in regions:
        mec_ts = aggregate_region('MEC', region, return_time_series=True)
        ssa_ts = aggregate_region('SSA', region, return_time_series=True)
        mac_ts = aggregate_region('MAC', region, return_time_series=True)
        frames = []
        for model in sorted(mec_ts.keys()):
            if model not in ssa_ts:
                continue
            mec = mec_ts[model]
            df_m = pd.DataFrame({
                'model': model,
                'time': pd.to_datetime(mec.time.values),
                'MEC': mec.values,
            })
            df_m['SSA'] = ssa_ts[model].values
            if model in mac_ts:
                df_m['MAC'] = mac_ts[model].values
            frames.append(df_m)
        monthly[region] = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return monthly


region_monthly = build_monthly_region_data(REGIONS.keys())
for region, df in region_monthly.items():
    print(f"{region} monthly rows: {len(df)} (models x months)")


## 6. Monthly regional plots (no temporal mean)

Spatial area-weighted regional mean at each month (fire-season windows per region).
Each point is one model in one month.


In [ ]:
# --make monthly plot with different colors for each month --

if region_monthly:
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    for idx, (region, df) in enumerate(region_monthly.items()):
        if df.empty or 'SSA' not in df.columns:
            continue
        ax = axes[idx]
        ax.scatter(df['SSA'], df['MEC'], s=35, alpha=0.35, color=colors[idx], edgecolors='none')
        ax.set_xlabel('SSA', fontweight='bold')
        ax.set_ylabel('MEC (m² g⁻¹)', fontweight='bold')
        ax.set_title(f'{region.upper()} — monthly points', fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        n_models = df['model'].nunique()
        n_months = df['time'].nunique()
        ax.text(0.03, 0.97, f'{n_models} models × {n_months} months', transform=ax.transAxes,
                va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle('MEC vs SSA — monthly regional values (spatial mean only)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    out = Path(project_root) / 'notebooks' / 'SSA_MEC_scatter_monthly.png'
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out.name}')

    # Per-region monthly time series (regional mean MEC and SSA vs time)
    for region, df in region_monthly.items():
        if df.empty or 'SSA' not in df.columns:
            continue
        ts = df.groupby('time')[['MEC', 'SSA']].mean().sort_index()
        fig, ax1 = plt.subplots(figsize=(10, 4))
        ax1.plot(ts.index, ts['MEC'], 'o-', color='steelblue', label='MEC')
        ax1.set_ylabel('MEC (m² g⁻¹)', color='steelblue', fontweight='bold')
        ax2 = ax1.twinx()
        ax2.plot(ts.index, ts['SSA'], 's--', color='darkorange', label='SSA')
        ax2.set_ylabel('SSA', color='darkorange', fontweight='bold')
        ax1.set_xlabel('Time', fontweight='bold')
        ax1.set_title(f'{region.upper()} — multi-model mean monthly time series', fontweight='bold')
        ax1.grid(True, alpha=0.3)
        plt.tight_layout()
        out_ts = Path(project_root) / 'notebooks' / f'SSA_MEC_monthly_timeseries_{region}.png'
        plt.savefig(out_ts, dpi=300, bbox_inches='tight')
        plt.show()
        print(f'Saved: {out_ts.name}')
else:
    print('No monthly regional data to plot')


In [ ]:
if region_monthly:
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    for idx, (region, df) in enumerate(region_monthly.items()):
        if df.empty or 'SSA' not in df.columns:
            continue
        ax = axes[idx]
        ax.scatter(df['SSA'], df['MEC'], s=35, alpha=0.35, color=colors[idx], edgecolors='none')
        ax.set_xlabel('SSA', fontweight='bold')
        ax.set_ylabel('MEC (m² g⁻¹)', fontweight='bold')
        ax.set_title(f'{region.upper()} — monthly points', fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        n_models = df['model'].nunique()
        n_months = df['time'].nunique()
        ax.text(0.03, 0.97, f'{n_models} models × {n_months} months', transform=ax.transAxes,
                va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle('MEC vs SSA — monthly regional values (spatial mean only)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    out = Path(project_root) / 'notebooks' / 'SSA_MEC_scatter_monthly.png'
    plt.savefig(out, dpi=300, bbox_inches='tight')
    
    plt.show()
    print(f'Saved: {out.name}')

    # Per-region monthly time series (regional mean MEC and SSA vs time)
    for region, df in region_monthly.items():
        if df.empty or 'SSA' not in df.columns:
            continue
        ts = df.groupby('time')[['MAC', 'SSA']].mean().sort_index()
        fig, ax1 = plt.subplots(figsize=(10, 4))
        ax1.plot(ts.index, ts['MAC'], 'o-', color='steelblue', label='MAC')
        ax1.set_ylabel('MAC (m² g⁻¹)', color='steelblue', fontweight='bold')
        ax2 = ax1.twinx()
        ax2.plot(ts.index, ts['SSA'], 's--', color='darkorange', label='SSA')
        ax2.set_ylabel('SSA', color='darkorange', fontweight='bold')
        ax1.set_xlabel('Time', fontweight='bold')
        ax1.set_title(f'{region.upper()} — multi-model mean monthly time series', fontweight='bold')
        ax1.grid(True, alpha=0.3)
        plt.tight_layout()
        out_ts = Path(project_root) / 'notebooks' / f'SSA_MAC_monthly_timeseries_{region}.png'
        plt.savefig(out_ts, dpi=300, bbox_inches='tight')
        plt.show()
        print(f'Saved: {out_ts.name}')
else:
    print('No monthly regional data to plot')


## 7. Regional-mean scatter plots (one point per model)


### THEORY 
$$
\begin{align*}
MAC &= \frac{AAOD}{Col\_Density} \\
L^2 M^{-1} &= [-] (M L^{-2})^{-1} \\
AAOD &= AOD (1-SSA) \ [unitless]\\
MAC  &= \frac{AOD (1-SSA)}{Col\_Density} \\
MEC  &= \frac{AOD}{Col\_Density} \\
MAC  &=  MEC (1-SSA)
\end{align*}
$$

MEC is effectiveness in extinction per unit mass of aerosol, which assumed to be primarily determined by aersol mixture, and aerosol size (amount of mie scattering). For absorption, the main compoenent should come from BC/EC and OM ratio and secondarily Dust, the secondary effect should come from hygroscopicity. Therefore the relation between MAC and SSA should mainly depending on aerosol composition and mixing state. Since model often assum similar BC/OM ratio for emission, the MAC for specific type of aerosol mixture like fire, seasalt and dust should have a good statics correlation. Based on this relation, a regression should fit without intercept and with 1-ssa. 

what did i miss? A bad correspondance to MEC value with the slope means SSA is not sensitive in the data. Explaination? Might be the best to not look globally anyway. Why not filtering land and sea quickly? check the SPEXone data? on SSA? and land sea mask + 250km away from the land and in pacific? that could be a good one. 


In [ ]:
if region_data:
    # Create figure with subplots (2x2 grid)
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    for idx, (region, df) in enumerate(region_data.items()):
        ax = axes[idx]
        
        # Create scatter plot
        scatter = ax.scatter(df['SSA'], df['MEC'], 
                            s=100, alpha=0.6, color=colors[idx],
                            edgecolors='black', linewidth=1.5)
        
        # Add labels for each point (model name)
        for i, model in enumerate(df['model']):
            ax.annotate(model.replace('_', '\\n')[:15], 
                        (df['SSA'].iloc[i], df['MEC'].iloc[i]),
                        fontsize=8, alpha=0.7, 
                        xytext=(5, 5), textcoords='offset points')
        
        # Labels and title
        ax.set_xlabel('Single Scattering Albedo (SSA)', fontsize=11, fontweight='bold')
        ax.set_ylabel('Mass Extinction Coefficient (MEC)', fontsize=11, fontweight='bold')
        ax.set_title(f'{region.upper()} Region', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        
        # Add statistics box
        stats_text = f"n={len(df)}\nMEC: {df['MEC'].mean():.2f}±{df['MEC'].std():.2f}\nSSA: {df['SSA'].mean():.2f}±{df['SSA'].std():.2f}"
        ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
               fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle('MEC vs SSA across Different Regions', fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(Path(project_root) / 'notebooks' / 'SSA_MEC_scatter_plot.png', 
                dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Scatter plot saved as 'SSA_MEC_scatter_plot.png'")
else:
    print("✗ No region data available for plotting")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import linregress

def plot_combined_regions(region_data, x_col, y_col, x_label, y_label, 
                          save_fig=False, filename="plot.png", 
                          use_model_numbers=False, force_intercept_zero=False):
    """
    Plots regions with regression stats inside and a wrapped model legend below the x-axis.
    Includes option to force the linear regression intercept through the origin (0,0).
    """
    fig, ax = plt.subplots(figsize=(10, 7)) 
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    model_col = 'model' if 'model' in next(iter(region_data.values())).columns else 'model_name'
    all_models = sorted(list(set().union(*[df[model_col].values for df in region_data.values()])))

    for idx, (region, df) in enumerate(region_data.items()):
        color = colors[idx % len(colors)]
        x_data = df[x_col].values
        y_data = df[y_col].values
        
        # Plot Scatter
        ax.scatter(x_data, y_data, s=90, alpha=0.6, color=color, label=f"{region.upper()}")
        
        if use_model_numbers:
            for _, row in df.iterrows():
                model_idx = all_models.index(row[model_col]) + 1
                ax.annotate(str(model_idx), (row[x_col], row[y_col]), 
                            xytext=(5, 5), textcoords="offset points", fontsize=8, color=color)
        
        # --- Regression Engine ---
        if force_intercept_zero:
            # Force intercept to 0: y = mx
            # m = sum(x*y) / sum(x^2)
            slope = np.sum(x_data * y_data) / np.sum(x_data**2)
            line = slope * x_data
            
            # Uncentered R^2 calculation
            residuals = y_data - line
            r_squared = 1 - (np.sum(residuals**2) / np.sum(y_data**2))
            
            # Equation formatting
            eq_str = f"y = {slope:.2f}x"
            r2_label = f"Uncentered $R^2$"
        else:
            # Standard Ordinary Least Squares (OLS)
            slope, intercept, r_value, _, _ = linregress(x_data, y_data)
            line = slope * x_data + intercept
            r_squared = r_value**2
            
            # Equation formatting
            sign = "+" if intercept >= 0 else "-"
            eq_str = f"y = {slope:.2f}x {sign} {abs(intercept):.2f}"
            r2_label = f"$R^2$"
            
        ax.plot(x_data, line, color=color, linestyle='--', alpha=0.8)
        
        # Combined stats label box
        text_label = f"{region.upper()}: {eq_str}\n{r2_label} = {r_squared:.2f}"
        ax.text(0.02, 0.94 - (idx * 0.12), text_label, transform=ax.transAxes, 
                color=color, fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8, edgecolor=color))

    ax.set_xlabel(x_label, fontsize=12, fontweight='bold')
    ax.set_ylabel(y_label, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(title="Regions", loc="upper right")
    
    if use_model_numbers:
        legend_elements = [plt.Line2D([0], [0], marker='', color='w', label=f"{i+1}: {model}") 
                           for i, model in enumerate(all_models)]
        ax.legend(handles=legend_elements, title="Model Index Key", 
                  loc='upper center', bbox_to_anchor=(0.5, -0.15), 
                  fontsize=8, title_fontsize=9, ncol=2)

    # Reserves room at the bottom for the model index wrapper
    plt.subplots_adjust(bottom=0.25)
    
    if save_fig:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"Saved: {filename}")
        
    plt.show()
# Setup data column
for reg in region_data:
    region_data[reg]['1-SSA'] = 1 - region_data[reg]['SSA']

# Plot 1: Standard with saving enabled
plot_combined_regions(
    region_data, 
    x_col='SSA', 
    y_col='MAC', 
    x_label='SSA', 
    y_label='MAC $m^2 g^{-1}$',
    save_fig=False, 
    use_model_numbers=False
)
plot_combined_regions(
    region_data, 
    x_col='1-SSA', 
    y_col='MAC', 
    x_label='1-SSA', 
    y_label='MAC $m^2 g^{-1}$',
    save_fig=False,
    filename="MAC_vs_1-SSA_regions.png",
    use_model_numbers=False
)
plot_combined_regions(
    region_data, 
    x_col='1-SSA', 
    y_col='MAC', 
    x_label='1-SSA', 
    y_label='MAC $m^2 g^{-1}$',
    save_fig=False,
    filename="MAC_vs_1-SSA_regions.png",
    force_intercept_zero= True,
    use_model_numbers=True
)
# plot_combined_regions(
#     region_data, 
#     x_col='SSA', 
#     y_col='MEC', 
#     x_label='SSA', 
#     y_label='MEC $m^2 g^{-1}$',
#     save_fig=False,
#     use_model_numbers=True
# )

## 9. Individual high-quality plots per region


In [ ]:
if region_data:
    for region, df in region_data.items():
        fig, ax = plt.subplots(figsize=(10, 7))
        
        # Remove NaNs/infs that can break the regression
        plot_df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['SSA', 'MEC'])
        if plot_df.empty:
            print(f'No valid data for {region}; skipping plot.')
            plt.close(fig)
            continue
        
        # Create scatter plot
        scatter = ax.scatter(plot_df['SSA'], plot_df['MEC'],
                            s=150, alpha=0.7, color='steelblue',
                            edgecolors='darkblue', linewidth=2)
        
        # Add trend line only if there is enough variation
        if len(plot_df) > 1 and plot_df['SSA'].nunique() > 1:
            z = np.polyfit(plot_df['SSA'], plot_df['MEC'], 1)
            p = np.poly1d(z)
            x_trend = np.linspace(plot_df['SSA'].min(), plot_df['SSA'].max(), 100)
            ax.plot(x_trend, p(x_trend), "r--", alpha=0.8, linewidth=2, label='Trend line')
            ax.legend(loc='best', fontsize=10)
        
        # Add labels for each point
        for i, model in enumerate(plot_df['model']):
            ax.annotate(model,
                        (plot_df['SSA'].iloc[i], plot_df['MEC'].iloc[i]),
                        fontsize=9, alpha=0.8,
                        xytext=(8, 8), textcoords='offset points',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))
        
        # Calculate correlation
        correlation = plot_df['SSA'].corr(plot_df['MEC'])
        
        # Labels and title
        ax.set_xlabel('Single Scattering Albedo (SSA)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Mass Extinction Coefficient (MEC)', fontsize=12, fontweight='bold')
        ax.set_title(f'MEC vs SSA - {region.upper()} Region', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        
        # Add statistics box
        stats_text = (
            f"Statistics:\n"
            f"n = {len(plot_df)} models\n"
            f"Correlation (r) = {correlation:.3f}\n"
            f"\nMEC:\n"
            f"  Mean = {plot_df['MEC'].mean():.3f}\n"
            f"  Std = {plot_df['MEC'].std():.3f}\n"
            f"  Min = {plot_df['MEC'].min():.3f}\n"
            f"  Max = {plot_df['MEC'].max():.3f}\n"
            f"\nSSA:\n"
            f"  Mean = {plot_df['SSA'].mean():.3f}\n"
            f"  Std = {plot_df['SSA'].std():.3f}\n"
            f"  Min = {plot_df['SSA'].min():.3f}\n"
            f"  Max = {plot_df['SSA'].max():.3f}"
        )
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
               fontsize=10, verticalalignment='top', family='monospace',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
        
        plt.tight_layout()
        plot_filename = Path(project_root) / 'notebooks' / f'SSA_MEC_scatter_{region}.png'
        plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"Plot saved for {region}: {plot_filename.name}")

## 10. Summary statistics table


In [ ]:
if region_data:
    print("\n" + "="*80)
    print("SUMMARY STATISTICS ACROSS REGIONS")
    print("="*80 + "\n")
    
    summary_stats = []
    
    for region, df in region_data.items():
        correlation = df['SSA'].corr(df['MEC'])
        summary_stats.append({
            'Region': region.upper(),
            'N_Models': len(df),
            'MEC_Mean': f"{df['MEC'].mean():.4f}",
            'MEC_Std': f"{df['MEC'].std():.4f}",
            'SSA_Mean': f"{df['SSA'].mean():.4f}",
            'SSA_Std': f"{df['SSA'].std():.4f}",
            'Correlation': f"{correlation:.4f}"
        })
    
    summary_df = pd.DataFrame(summary_stats)
    print(summary_df.to_string(index=False))
    
    # Save summary to CSV
    csv_path = Path(project_root) / 'notebooks' / 'SSA_MEC_summary_statistics.csv'
    summary_df.to_csv(csv_path, index=False)
    print(f"\n✓ Summary statistics saved to: {csv_path.name}")

## 11. Export detailed model data


In [ ]:
if region_data:
    print("\nDetailed Model Data by Region:")
    print("="*80 + "\n")
    
    for region, df in region_data.items():
        print(f"\n{region.upper()} REGION:")
        print("-" * 80)
        
        # Sort by MEC value
        df_sorted = df.sort_values('MEC', ascending=False)
        
        # Display table
        display_df = df_sorted.copy()
        display_df['SSA'] = display_df['SSA'].round(4)
        display_df['MEC'] = display_df['MEC'].round(4)
        print(display_df.to_string(index=False))
        
        # Save to CSV
        csv_path = Path(project_root) / 'notebooks' / f'SSA_MEC_data_{region}.csv'
        df_sorted.to_csv(csv_path, index=False)
        print(f"✓ Data saved to: {csv_path.name}")

## 12. Cross-regional comparison


In [ ]:
if region_data and len(region_data) > 1:
    # Create a comparison plot across regions
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    regions_list = list(region_data.keys())
    
    # MEC comparison
    ax1 = axes[0]
    for i, region in enumerate(regions_list):
        df = region_data[region]
        ax1.boxplot([df['MEC']], positions=[i], widths=0.6, patch_artist=True,
                    boxprops=dict(facecolor=plt.cm.Set3(i), alpha=0.7))
    ax1.set_xticklabels([r.upper() for r in regions_list])
    ax1.set_ylabel('Mass Extinction Coefficient (MEC)', fontsize=11, fontweight='bold')
    ax1.set_title('MEC Distribution Across Regions', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # SSA comparison
    ax2 = axes[1]
    for i, region in enumerate(regions_list):
        df = region_data[region]
        ax2.boxplot([df['SSA']], positions=[i], widths=0.6, patch_artist=True,
                    boxprops=dict(facecolor=plt.cm.Set3(i), alpha=0.7))
    ax2.set_xticklabels([r.upper() for r in regions_list])
    ax2.set_ylabel('Single Scattering Albedo (SSA)', fontsize=11, fontweight='bold')
    ax2.set_title('SSA Distribution Across Regions', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Cross-Regional Comparison of MEC and SSA', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plot_filename = Path(project_root) / 'notebooks' / 'SSA_MEC_cross_regional_comparison.png'
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Cross-regional comparison plot saved: {plot_filename.name}")

## 13. Summary and notes


In [ ]:
print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print(f"\nProject Root: {project_root}")
print(f"Monthly Pickle: {monthly_pickle}")
print(f"Processed Monthly Directory: {processed_monthly_dir}")
print(f"Land Mask Used: {LAND_MASK_PATH}")
print(f"Surface Type: {SURFACE_TYPE}")
print(f"\nRegions Analyzed: {', '.join([r.upper() for r in region_data.keys()])}")
print(f"Total Models: {sum(len(df) for df in region_data.values())}")
print("\nOutput Files Generated:")
output_dir = Path(project_root) / 'notebooks'
for output_file in sorted(output_dir.glob('SSA_MEC*.png')) + sorted(output_dir.glob('SSA_MEC*.csv')):
    print(f"  {output_file.name}")
print("\nNote: All plots and data files are saved in the notebooks directory.")
print("="*80)
